# 03. Feature Selection

This notebook covers the third step in a typical QSAR workflow:
- Loading the featurized dataset
- Applying feature selection techniques to identify the most relevant descriptors
- Reducing dimensionality to avoid overfitting and improve model interpretability

ProQSAR provides built-in feature selection methods including variance-based, correlation-based, and model-based selection.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from proqsar.Model.FeatureSelector.feature_selector import FeatureSelector
from proqsar.Config.config import Config
from sklearn.preprocessing import StandardScaler

# Set style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 3.1 Load Featurized Dataset

Load the featurized dataset from the previous step.

In [ ]:
# Load the featurized dataset
data_path = '../Project/featurized_data.csv'
df = pd.read_csv(data_path)

print(f"Dataset loaded: {df.shape}")
print(f"Number of compounds: {len(df)}")
print(f"Number of features: {df.shape[1] - 2}")  # Subtract ID and activity columns
df.head()

## 3.2 Prepare Data for Feature Selection

Separate features from the target variable and standardize if needed.

In [ ]:
# Separate features and target
X = df.drop(['Smiles', 'pChEMBL'], axis=1)
y = df['pChEMBL']
ids = df['Smiles']

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print(f"\nFeature columns (first 10):")
print(X.columns.tolist()[:10])

## 3.3 Configure Feature Selector

ProQSAR supports various feature selection methods:
- **Variance-based**: Remove low-variance features
- **Correlation-based**: Remove highly correlated features
- **Model-based**: Select features based on model importance (e.g., RandomForest, Lasso)
- **Sequential selection**: Forward/backward feature selection

In [ ]:
# Configure feature selector
config = Config(
    feature_selector={
        "method": "tree",  # Options: "tree", "lasso", "mutual_info", "none"
        "n_features_to_select": 50,  # Select top 50 features
    }
)

print("Feature selector configured!")
print(f"Method: {config.feature_selector_config['method']}")
print(f"Target number of features: {config.feature_selector_config['n_features_to_select']}")

## 3.4 Initialize and Fit Feature Selector

Create and fit the feature selector to identify important features.

In [ ]:
# Initialize feature selector
feature_selector = FeatureSelector(
    activity_col="pChEMBL",
    id_col="Smiles",
    config=config,
    save_dir="../Project/FeatureSelector",
)

# Fit the feature selector
print("Fitting feature selector...")
selected_df = feature_selector.fit_transform(df)

print(f"\nFeature selection complete!")
print(f"Original features: {X.shape[1]}")
print(f"Selected features: {selected_df.shape[1] - 2}")  # Subtract ID and activity

## 3.5 Analyze Selected Features

Examine the importance and characteristics of selected features.

In [ ]:
# Get selected feature names
selected_features = [col for col in selected_df.columns if col not in ['Smiles', 'pChEMBL']]

print(f"Selected features ({len(selected_features)}):")
print(selected_features[:20])  # Show first 20

In [ ]:
# If feature importances are available, plot them
if hasattr(feature_selector, 'feature_importances_') and feature_selector.feature_importances_ is not None:
    importances = feature_selector.feature_importances_
    
    # Create DataFrame for plotting
    importance_df = pd.DataFrame({
        'feature': selected_features,
        'importance': importances[:len(selected_features)]
    }).sort_values('importance', ascending=False)
    
    # Plot top 20 features
    plt.figure(figsize=(12, 8))
    plt.barh(range(20), importance_df['importance'].head(20))
    plt.yticks(range(20), importance_df['feature'].head(20))
    plt.xlabel('Feature Importance')
    plt.ylabel('Feature')
    plt.title('Top 20 Most Important Features')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("Feature importances not available for this selection method.")

In [ ]:
# Analyze correlation among selected features
X_selected = selected_df[selected_features]
correlation_matrix = X_selected.corr()

# Plot correlation heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, cmap='coolwarm', center=0, 
            vmin=-1, vmax=1, square=True, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix of Selected Features')
plt.tight_layout()
plt.show()

# Count highly correlated feature pairs
high_corr = (correlation_matrix.abs() > 0.8) & (correlation_matrix.abs() < 1.0)
n_high_corr = high_corr.sum().sum() // 2
print(f"\nNumber of highly correlated feature pairs (|r| > 0.8): {n_high_corr}")

In [ ]:
# Analyze correlation between selected features and activity
feature_activity_corr = X_selected.corrwith(y).abs().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
feature_activity_corr.head(20).plot(kind='barh')
plt.xlabel('Absolute Correlation with Activity')
plt.ylabel('Feature')
plt.title('Top 20 Features Most Correlated with Activity')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(f"\nMean absolute correlation with activity: {feature_activity_corr.mean():.4f}")
print(f"Max absolute correlation with activity: {feature_activity_corr.max():.4f}")

## 3.6 Save Selected Features Dataset

Save the dataset with selected features for use in modeling.

In [ ]:
# Save selected features dataset
output_path = '../Project/selected_features_data.csv'
selected_df.to_csv(output_path, index=False)

print(f"Selected features dataset saved to: {output_path}")
print(f"Shape: {selected_df.shape}")

# Also save the list of selected features
feature_list_path = '../Project/selected_features_list.txt'
with open(feature_list_path, 'w') as f:
    for feature in selected_features:
        f.write(f"{feature}\n")
        
print(f"Selected features list saved to: {feature_list_path}")

## 3.7 Summary

Summarize the feature selection process.

In [ ]:
print("="*60)
print("FEATURE SELECTION SUMMARY")
print("="*60)
print(f"Original features: {X.shape[1]}")
print(f"Selected features: {len(selected_features)}")
print(f"Reduction: {(1 - len(selected_features) / X.shape[1]) * 100:.1f}%")
print(f"\nSelection method: {config.feature_selector_config['method']}")
print(f"Highly correlated pairs: {n_high_corr}")
print(f"Mean |correlation| with activity: {feature_activity_corr.mean():.4f}")
print(f"\nData saved to: {output_path}")
print("\nDataset is ready for splitting and modeling!")
print("="*60)

## Next Steps

The most relevant features have been selected. The next notebook (04_dataset_splitting.ipynb) will:
- Split the dataset into training and test sets
- Use appropriate splitting strategies (random, scaffold-based, Kennard-Stone)
- Ensure proper representation of chemical diversity in both sets